In [1]:
import pandas as pd

In [2]:
#loading the data
df = pd.read_csv('DimItemL1.csv', encoding='latin1')
pd.set_option('display.max_rows', None)
df.head()


,Item,Short Description,Description,Primary Barcode,Season,Item PPG,Item Sub Category,Item Category,Jobber,Weight Item,...,Height,Dimension UOM,Weight,Weight UOM,Volume,Volume UOM,Critical Dimension 1,Critical Dimension 2,Critical Dimension 3,Column1
0,100000066,TRAYBAKE,100000066 AB CHOC TRAYBAKE CHOC EACH,100000066,NaN,320,300022,3000,NaN,False,...,NaN,cm,0.880,kg,1.665,L,NaN,NaN,NaN,NaN
1,100000077,CAKES,100000077 KIPLING FANCIES FRENCH 8PK,100000077,NaN,305,300021,3000,NaN,False,...,NaN,cm,0.272,kg,1.033,L,NaN,NaN,NaN,NaN
2,100000136,CAKES,100000136 BALCONI MIX MILK 10PK,100000136,NaN,545,300021,3000,NaN,False,...,NaN,cm,0.425,kg,2.364,L,NaN,NaN,NaN,NaN
3,100000620,MUFFINS,100000620 ASDA MINI MUFFINS DCHOC 16PK,100000620,NaN,835,300010,3000,NaN,False,...,NaN,cm,0.417,kg,3.611,L,NaN,NaN,NaN,NaN
4,100001490,DELI WRAPS,100001490 MISSION DELI WRAPS PLAIN 6PK,100001490,NaN,247,859918,8599,NaN,False,...,NaN,cm,0.418,kg,1.701,L,NaN,NaN,NaN,NaN


In [3]:
#check columns with no missing values
# Identify columns with no missing values to find usable fields
df.isnull().sum()[df.isnull().sum() == 0]


Item                                                         0
Description                                                  0
Primary Barcode                                              0
Item PPG                                                     0
Item Sub Category                                            0
Item Category                                                0
Weight Item                                                  0
Food Traceability Lot Code Tracking                          0
Derive CPD From Expiration Date minus Shelf Days?            0
Calculate a missing expiry or manufacturing date?            0
Evaluate Asset Determination                                 0
Validate inventory being received for older expiry dates?    0
Derive CPD From Creation Date plus Product Life?             0
Derive Expiry Date From Creation Date plus Product Life?     0
Maximum Lpn Quantity                                         0
Minimum receive to expire days                         

In [4]:

# Select relevant columns
wanted = ['Item', 'Description', 'Merchandizing Department ID', 'Merchandizing Department ID_3', 'Merchandise Group']
df_clean = df[wanted].copy()

# Rename columns
df_clean.columns = ['item_id', 'description', 'department', 'department_id', 'merchandise_group']

# Clean description: remove leading item number, fix spacing, title case
df_clean['description'] = (df_clean['description']
    .str.replace(r'^\d+\s+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)

# Fix data types
df_clean['item_id'] = df_clean['item_id'].astype(int)
df_clean['department_id'] = df_clean['department_id'].astype(int)

# Validate
print(f"Duplicate items: {df_clean.duplicated(subset='item_id').sum()}")
print(f"Nulls:\n{df_clean.isnull().sum()}")
df_clean.head()


Duplicate items: 0
Nulls:
item_id              0
description          0
department           0
department_id        0
merchandise_group    0
dtype: int64


,item_id,description,department,department_id,merchandise_group
0,100000066,Ab Choc Traybake Choc Each,Bakery Bought In,39,Z3 - Chilled 3
1,100000077,Kipling Fancies French 8Pk,Bakery Bought In,39,Z3 - Chilled 3
2,100000136,Balconi Mix Milk 10Pk,Bakery Bought In,39,Z6 - Ambient Grocery
3,100000620,Asda Mini Muffins Dchoc 16Pk,Bakery In Store,62,Z3 - Chilled 3
4,100001490,Mission Deli Wraps Plain 6Pk,Bakery Bought In,81,Z3 - Chilled 3


In [5]:
# Load department reference table
dept = pd.read_csv('ItemDepartment.csv')
dept.columns = dept.columns.str.strip()
dept.rename(columns={'Item Department': 'department_id', 'Description': 'department_name'}, inplace=True)

# Left join on department_id
df_clean = df_clean.merge(dept, on='department_id', how='left')

print(f"Unmatched rows: {df_clean['department_name'].isnull().sum()}")
df_clean.head()


Unmatched rows: 0


,item_id,description,department,department_id,merchandise_group,department_name
0,100000066,Ab Choc Traybake Choc Each,Bakery Bought In,39,Z3 - Chilled 3,Cakes & Chilled BI
1,100000077,Kipling Fancies French 8Pk,Bakery Bought In,39,Z3 - Chilled 3,Cakes & Chilled BI
2,100000136,Balconi Mix Milk 10Pk,Bakery Bought In,39,Z6 - Ambient Grocery,Cakes & Chilled BI
3,100000620,Asda Mini Muffins Dchoc 16Pk,Bakery In Store,62,Z3 - Chilled 3,Cakes IS
4,100001490,Mission Deli Wraps Plain 6Pk,Bakery Bought In,81,Z3 - Chilled 3,Bread & Morn Goods BI


In [6]:
# Load facilities reference table
facilities = pd.read_csv('ItemFacilities.csv')
facilities.columns = facilities.columns.str.strip()

# Rename merge key, then merge on item_id
facilities = facilities.rename(columns={'Item ID': 'item_id'})

df_clean = df_clean.merge(
    facilities[['item_id', 'Unit Price']],
    on='item_id',
    how='left'
)

print(f"Unmatched rows: {df_clean['Unit Price'].isnull().sum()}")
df_clean.head()



Unmatched rows: 0


,item_id,description,department,department_id,merchandise_group,department_name,Unit Price
0,100000066,Ab Choc Traybake Choc Each,Bakery Bought In,39,Z3 - Chilled 3,Cakes & Chilled BI,4.44
1,100000077,Kipling Fancies French 8Pk,Bakery Bought In,39,Z3 - Chilled 3,Cakes & Chilled BI,1.67
2,100000136,Balconi Mix Milk 10Pk,Bakery Bought In,39,Z6 - Ambient Grocery,Cakes & Chilled BI,1.13
3,100000620,Asda Mini Muffins Dchoc 16Pk,Bakery In Store,62,Z3 - Chilled 3,Cakes IS,1.48
4,100001490,Mission Deli Wraps Plain 6Pk,Bakery Bought In,81,Z3 - Chilled 3,Bread & Morn Goods BI,0.66


In [7]:
df_clean.to_csv('DimItemL1_cleaned.csv', index=False)
print("Saved successfully.")

Saved successfully.
